# Executive Budget & Performance Analytics Dashboard

## Project Objective

This project simulates the financial performance of a B2B SaaS company.

The goal is to create a realistic FP&A dataset that supports:

- Revenue Analysis
- Profitability Analysis
- Budget vs Actual Analysis
- Customer Growth Analysis
- Variance Analysis

The dataset intentionally includes business events that explain financial performance changes.

This dataset is designed to replicate real-world FP&A decision-making scenarios in a SaaS business environment.

## Business Scenario

### Company

InsightPro Analytics

### Industry

B2B SaaS

### Business Model

Customers subscribe to a cloud-based analytics platform.

Revenue is generated through recurring monthly subscriptions.

The simulation includes:

- New Customer Acquisition
- Customer Churn
- Seasonal Revenue Patterns
- Budget Miss Events

## Import Libraries

We use:

- pandas for data manipulation
- numpy for data generation

A fixed random seed ensures reproducibility.

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

## Define Analysis Period

The simulation covers January 2024 through December 2024.

In [2]:
months = pd.date_range(
    start="2024-01-01",
    end="2024-12-01",
    freq="MS"
)

## Define Subscription Plans

Monthly recurring subscription fees.

In [3]:
pricing = {
    "Basic": 500,
    "Professional": 1500,
    "Enterprise": 5000
}

## Generate Initial Customer Base

The company starts the year with 60 customers.

Each customer is assigned:

- Customer ID
- Region
- Subscription Plan

The customer table acts as a dimension table for later analysis.

In [4]:
customer_counter = 1000

active_customers = pd.DataFrame({
    "customer_id": range(
        customer_counter,
        customer_counter + 60
    ),
    "region": np.random.choice(
        ["North America", "Europe", "Asia Pacific"],
        60,
        p=[0.5, 0.3, 0.2]
    ),
    "plan": np.random.choice(
        ["Basic", "Professional", "Enterprise"],
        60,
        p=[0.5, 0.35, 0.15]
    )
})

customer_counter += 60

## Generate Monthly Transactions

Business Events Included:

1. New Customers
2. Customer Churn
3. Seasonal Growth
4. Budget Miss Events

These events will later support FP&A analysis.

In [5]:
transactions = []

for month in months:

    month_num = month.month

    # --------------------
    # New Customers
    # --------------------

    new_customers = np.random.randint(2, 6)

    new_df = pd.DataFrame({
        "customer_id": range(
            customer_counter,
            customer_counter + new_customers
        ),
        "region": np.random.choice(
            ["North America", "Europe", "Asia Pacific"],
            new_customers,
            p=[0.5, 0.3, 0.2]
        ),
        "plan": np.random.choice(
            ["Basic", "Professional", "Enterprise"],
            new_customers,
            p=[0.5, 0.35, 0.15]
        )
    })

    customer_counter += new_customers

    active_customers = pd.concat(
        [active_customers, new_df],
        ignore_index=True
    )

    # --------------------
    # Customer Churn
    # --------------------

    churn_count = np.random.randint(0, 3)

    if len(active_customers) > churn_count:

        churn_ids = np.random.choice(
            active_customers["customer_id"],
            churn_count,
            replace=False
        )

        active_customers = active_customers[
            ~active_customers["customer_id"].isin(churn_ids)
        ]

    # --------------------
    # Seasonality
    # --------------------

    seasonality_factor = 1.0

    if month_num in [10, 11, 12]:
        seasonality_factor = 1.15

    # --------------------
    # Budget Miss Months
    # --------------------

    budget_miss_factor = 1.0

    if month_num in [8, 11]:
        budget_miss_factor = 0.90

    # --------------------
    # Generate Revenue
    # --------------------

    for _, customer in active_customers.iterrows():

        base_revenue = pricing[
            customer["plan"]
        ]

        revenue = (
            base_revenue
            * seasonality_factor
            * budget_miss_factor
        )

        transactions.append([
            month,
            customer["customer_id"],
            customer["region"],
            customer["plan"],
            revenue
        ])

## Create Transaction Table

In [6]:
transactions = pd.DataFrame(
    transactions,
    columns=[
        "date",
        "customer_id",
        "region",
        "plan",
        "revenue"
    ]
)

## Generate Cost

Operating costs vary by customer activity and infrastructure usage.

Cost ranges between 35% and 60% of revenue.

In [7]:
transactions["cost"] = (
    transactions["revenue"]
    * np.random.uniform(
        0.35,
        0.60,
        len(transactions)
    )
).round(2)

## Calculate Profit

Profit is one of the primary FP&A performance metrics.

Formula: Profit = Revenue - Cost

In [8]:
transactions["profit"] = (
    transactions["revenue"]
    - transactions["cost"]
).round(2)

## Sort Transactions

Sorting improves readability for analysis.

In [9]:
transactions = transactions.sort_values(
    ["date", "customer_id"]
).reset_index(drop=True)

## Review Dataset

Validate generated revenue, cost, and profit values.

In [10]:
transactions.head()

,date,customer_id,region,plan,revenue,cost,profit
0,2024-01-01,1000,North America,Basic,500.0,270.67,229.33
1,2024-01-01,1001,Asia Pacific,Basic,500.0,187.87,312.13
2,2024-01-01,1002,Europe,Professional,1500.0,650.39,849.61
3,2024-01-01,1003,Europe,Basic,500.0,184.44,315.56
4,2024-01-01,1004,North America,Basic,500.0,269.16,230.84


## Generate Budget Table

Management establishes monthly revenue targets.

Targets assume steady growth throughout the year.

In [11]:
budget = pd.DataFrame({
    "month": months,
    "budget_revenue": np.linspace(
        130000,
        200000,
        len(months)
    ).round(0)
})

## Generate Budget Cost

In [12]:
budget["budget_cost"] = (
    budget["budget_revenue"]
    * 0.50
).round(0)

## Calculate Budget Profit

In [13]:
budget["budget_profit"] = (
    budget["budget_revenue"]
    - budget["budget_cost"]
)

## Review Budget Data

In [14]:
budget.head()

,month,budget_revenue,budget_cost,budget_profit
0,2024-01-01,130000.0,65000.0,65000.0
1,2024-02-01,136364.0,68182.0,68182.0
2,2024-03-01,142727.0,71364.0,71363.0
3,2024-04-01,149091.0,74546.0,74545.0
4,2024-05-01,155455.0,77728.0,77727.0


## Export Datasets

The datasets will be used for:

- SQL Analysis
- KPI Development
- Power BI Dashboard Creation

In [15]:
transactions.to_csv(
    "fact_transactions.csv",
    index=False
)

budget.to_csv(
    "fact_budget.csv",
    index=False
)

## Business Assumptions

This dataset was intentionally designed to simulate a realistic FP&A environment for a B2B SaaS company.

Key assumptions:

1. Revenue is generated through monthly recurring subscriptions (SaaS model).
2. Customers are segmented by pricing plan (Basic, Professional, Enterprise).
3. Revenue is influenced by customer lifecycle events including:
   - New customer acquisition
   - Customer churn
4. Revenue exhibits seasonal patterns, with higher performance in Q4 due to increased business activity and budget utilization.
5. Certain months experience budget miss events where actual performance falls below expected targets.
6. Costs vary based on customer usage, support intensity, and infrastructure load.
7. Budget targets are defined monthly and increase gradually to reflect growth expectations.
8. Actual performance can be compared against budget for variance analysis (Budget vs Actual).

These assumptions support:

- Revenue Trend Analysis
- Customer Growth & Retention Analysis
- Seasonality Analysis
- Budget Variance Analysis
- Profitability Analysis
- Executive-Level KPI Reporting

These assumptions are intentionally designed to enable driver-based analysis rather than purely descriptive reporting.